# Investigation 09: Non-Linear UMAP Projection & Differential Rule Analysis

This notebook shifts away from linear PCA and utilizes a non-linear manifold learning technique called **UMAP** (Uniform Manifold Approximation and Projection). 

Spatial rule data is often highly sparse and non-linear. UMAP is exceptionally good at handling this type of data, grouping FOVs with similar complex spatial architectures into distinct "islands" or clusters. 

Once the FOVs are clustered by UMAP, we perform a **Differential Rule Analysis** using statistical tests to identify exactly which biological rules are driving the differences between our clinical states (e.g., Control vs. Severe).

## 1. Imports & Configuration

Here we import necessary libraries and set the global parameters.

In [19]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import umap
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests
import os
import ast
import warnings

warnings.filterwarnings('ignore')

# --- Global Parameters ---
# The metric to use for constructing the feature matrix.
METRIC_TO_USE = 'Lift'
NO_SELF = True

# UMAP Parameters
UMAP_N_NEIGHBORS = 15  # Controls the balance between local and global structure
UMAP_MIN_DIST = 0.0    # 0.0 allows tight clinical clustering; avoids artificial spreading
UMAP_METRIC = 'cosine' # Cosine distance is significantly better for high-dimensional sparse rule data

# Path to the results
RESULTS_PATH = '../../results/full_run/weighted_fpgrowth/data/results_CN.csv'
DATA_DIR = '../../data/MIBIGutCsv'


## 2. Data Loading & Metadata

We load the algorithm results and merge the FOV and biopsy metadata so we can map clinical scores and organs to our data.

In [20]:
def load_and_merge_metadata():
    print("Loading results and metadata...")
    
    # 1. Load Rule Results
    if not os.path.exists(RESULTS_PATH):
        raise FileNotFoundError(f"Results file not found at {RESULTS_PATH}")
    df_results = pd.read_csv(RESULTS_PATH)
    
    # 2. Load Metadata
    df_fovs = pd.read_csv(os.path.join(DATA_DIR, 'fovs_metadata.csv'))
    df_biopsy = pd.read_csv(os.path.join(DATA_DIR, 'biopsy_metadata.csv'))
    
    # 3. Merge FOV with Biopsy Metadata
    df_fovs = pd.merge(
        df_fovs,
        df_biopsy[['Biopsy_ID', 'Pathological score', 'Clinical score', 'Localization']],
        left_on='Patient',
        right_on='Biopsy_ID',
        how='left'
    )
    
    # Helper functions to clean up labels
    def get_organ(row):
        if pd.notna(row.get("Localization")): return row["Localization"]
        cohort = str(row.get("Cohort", ""))
        if "Colon" in cohort: return "Colon"
        if "Duodenum" in cohort: return "Duodenum"
        return "Unknown"
        
    def get_clean_score(row, score_col):
        if pd.notna(row[score_col]):
            return str(row[score_col])
        if str(row['FOV']).startswith('S_'):
            return 'Control_S'
        return 'Control'

    # Apply label cleaners
    df_fovs["Organ"] = df_fovs.apply(get_organ, axis=1)
    df_fovs["Pathological score"] = df_fovs.apply(lambda row: get_clean_score(row, "Pathological score"), axis=1)
    df_fovs["Clinical score"] = df_fovs.apply(lambda row: get_clean_score(row, "Clinical score"), axis=1)
    
    print(f"Loaded {len(df_results)} raw rules across {df_fovs['FOV'].nunique()} FOVs.")
    return df_results, df_fovs

df_results, df_fovs = load_and_merge_metadata()


Loading results and metadata...
Loaded 16146 raw rules across 288 FOVs.


## 3. Matrix Construction & Preprocessing

We parse the rule strings, remove self-referential rules (if configured), and build a matrix where rows are FOVs and columns are biological rules.

In [21]:
def construct_feature_matrix(df_results):
    print("Cleaning rule names and removing redundancies...")
    
    def clean_items(item_list_str):
        try:
            items = ast.literal_eval(item_list_str)
            cleaned = [item.replace('_CENTER', '').replace('_NEIGHBOR', '') for item in items]
            return ", ".join(sorted(cleaned))
        except Exception:
            return item_list_str

    # Clean names for readability
    df_results['Ant_Clean'] = df_results['Antecedents'].apply(clean_items)
    df_results['Con_Clean'] = df_results['Consequents'].apply(clean_items)
    df_results['Clean_Rule'] = df_results['Ant_Clean'] + " -> " + df_results['Con_Clean']

    # Remove self-rules (e.g., A -> A) if requested
    if NO_SELF:
        def has_overlap(row):
            ant_clean = set(row['Ant_Clean'].split(', '))
            con_clean = set(row['Con_Clean'].split(', '))
            return not ant_clean.isdisjoint(con_clean)

        overlap_mask = df_results.apply(has_overlap, axis=1)
        df_results = df_results[~overlap_mask].reset_index(drop=True)

    print("Pivoting into FOV x Rule matrix...")
    # Ensure we only have one value per FOV/Rule pair
    df_pivot = df_results.drop_duplicates(subset=['FOV', 'Clean_Rule']) 
    feature_matrix = df_pivot.pivot(index='FOV', columns='Clean_Rule', values=METRIC_TO_USE)

    # Fill missing values logically
    # For Lift, a value of 1.0 means 'no association' (neutral)
    fill_value = 1.0 if METRIC_TO_USE in ['Lift', 'Conviction'] else 0.0
    feature_matrix = feature_matrix.fillna(fill_value)
    
    # Replace any infinities with NaN, then replace NaNs with slightly above max value
    feature_matrix = feature_matrix.replace([np.inf, -np.inf], np.nan)
    max_val = feature_matrix.max().max()
    feature_matrix = feature_matrix.fillna(max_val * 1.1 if not pd.isna(max_val) else fill_value)

    # 1. Filter out rare rules (Curse of Dimensionality)
    # Keep only rules that appear in at least 10% of the unique FOVs
    rule_counts = (feature_matrix != fill_value).sum(axis=0)
    common_rules = rule_counts[rule_counts > len(feature_matrix) * 0.1].index
    feature_matrix = feature_matrix[common_rules]
    print(f"Rules remaining after filtering rare (<10% FOVs) rules: {len(feature_matrix.columns)}")

    # 2. Symmetric non-linear transform
    if METRIC_TO_USE in ['Lift', 'Conviction']:
        print("Applying log2 transformation for symmetric distances...")
        feature_matrix = np.log2(feature_matrix + 1e-9)

    # 3. Standardization 
    # Forces UMAP to capture systemic shifts across many rules rather than absolute variance of structural outliers
    from sklearn.preprocessing import StandardScaler
    print("Applying StandardScaler...")
    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(feature_matrix)
    feature_matrix = pd.DataFrame(scaled_data, index=feature_matrix.index, columns=feature_matrix.columns)

    print(f"Final matrix shape: {feature_matrix.shape}")
    return feature_matrix

feature_matrix = construct_feature_matrix(df_results)


Cleaning rule names and removing redundancies...
Pivoting into FOV x Rule matrix...
Rules remaining after filtering rare (<10% FOVs) rules: 127
Applying log2 transformation for symmetric distances...
Applying StandardScaler...
Final matrix shape: (288, 127)


## 4. UMAP Projection

We run the UMAP algorithm on our processed feature matrix to project the high-dimensional rules into a 2D space. FOVs that are close together in this 2D plot share similar structural and biological rule architectures.

In [22]:
def run_umap_projection(feature_matrix, df_fovs):
    print("Running UMAP projection...")
    
    # Initialize and fit UMAP
    reducer = umap.UMAP(
        n_neighbors=UMAP_N_NEIGHBORS,
        min_dist=UMAP_MIN_DIST,
        metric=UMAP_METRIC,
        random_state=42 # Set for reproducibility
    )
    
    umap_embedding = reducer.fit_transform(feature_matrix)
    
    # Create a DataFrame with the coordinates
    df_umap = pd.DataFrame(umap_embedding, index=feature_matrix.index, columns=['UMAP_1', 'UMAP_2'])
    
    # Merge metadata for plotting
    df_umap_merged = df_umap.reset_index().merge(
        df_fovs[['FOV', 'Organ', 'Clinical score', 'Pathological score']], 
        on='FOV', 
        how='left'
    )
    df_umap_merged['Organ'] = df_umap_merged['Organ'].fillna('Unknown')
    
    return df_umap_merged

df_umap_merged = run_umap_projection(feature_matrix, df_fovs)


Running UMAP projection...


In [23]:
def print_umap_plot(df_umap_merged, color_by='Organ'):
    
    # Define modern color palettes matching previous notebooks
    color_discrete_map = None
    category_orders = None
    
    if color_by in ['Pathological score', 'Clinical score']:
        color_discrete_map = {
            'Control_S': '#2E7D32', # dark green
            'Control': '#81C784',   # light green
            'Mild': '#FFA726',      # yellow/orange
            'Severe': '#D32F2F'     # red
        }
        category_orders = {color_by: ['Control_S', 'Control', 'Mild', 'Severe']}

    # Create Interactive Scatter Plot with Plotly
    fig = px.scatter(
        df_umap_merged, 
        x='UMAP_1', 
        y='UMAP_2', 
        color=color_by,
        color_discrete_map=color_discrete_map,
        category_orders=category_orders,
        hover_name='FOV',
        title=f'UMAP Projection of FOVs (Rule Metric: {METRIC_TO_USE})',
        subtitle=f'Color by: {color_by}',
        template='plotly_white',
        width=900,
        height=600
    )

    fig.update_traces(marker=dict(size=8, opacity=0.8, line=dict(width=1, color='DarkSlateGrey')))
    fig.show()


In [24]:
# Plot the UMAP projections
print_umap_plot(df_umap_merged, color_by='Organ')
print_umap_plot(df_umap_merged, color_by='Pathological score')


In [25]:
METRIC_TO_USE = 'Conviction'
NO_SELF = True

# Plot the UMAP projections
print_umap_plot(df_umap_merged, color_by='Organ')
print_umap_plot(df_umap_merged, color_by='Pathological score')


## 5. Differential Rule Analysis

Instead of looking at "loadings" (which are hard to interpret in non-linear spaces), we use a statistical approach. 

We ask a clear biological question: **"Which rules are statistically higher in group A compared to group B?"**
We use the Mann-Whitney U test (a robust non-parametric test) and correct for multiple testing using the Benjamini-Hochberg (FDR) method.

In [26]:
def run_differential_analysis(feature_matrix, metadata_df, target_col, group_a, group_b):
    """
    Identifies rules that are significantly different between two clinical groups.
    """
    print(f"Running Differential Analysis: {group_a} vs {group_b} (based on {target_col})")
    
    # 1. Identify FOVs belonging to each group
    fovs_a = metadata_df[metadata_df[target_col] == group_a]['FOV'].values
    fovs_b = metadata_df[metadata_df[target_col] == group_b]['FOV'].values
    
    # Ensure FOVs exist in our feature matrix
    fovs_a = [f for f in fovs_a if f in feature_matrix.index]
    fovs_b = [f for f in fovs_b if f in feature_matrix.index]
    
    if len(fovs_a) == 0 or len(fovs_b) == 0:
        print("Error: Not enough FOVs found for the specified groups.")
        return pd.DataFrame()

    # 2. Extract sub-matrices
    matrix_a = feature_matrix.loc[fovs_a]
    matrix_b = feature_matrix.loc[fovs_b]
    
    results = []
    
    # 3. Perform Mann-Whitney U Test for each rule
    for rule in feature_matrix.columns:
        vals_a = matrix_a[rule].values
        vals_b = matrix_b[rule].values
        
        # Only test if there's some variance
        if np.var(vals_a) == 0 and np.var(vals_b) == 0:
            continue
            
        try:
            stat, p_val = mannwhitneyu(vals_a, vals_b, alternative='two-sided')
            
            # Calculate Fold Change (Difference in means)
            mean_diff = np.mean(vals_a) - np.mean(vals_b)
            
            results.append({
                'Rule': rule,
                f'Mean_{group_a}': np.mean(vals_a),
                f'Mean_{group_b}': np.mean(vals_b),
                'Mean_Difference': mean_diff,
                'P_Value': p_val
            })
        except ValueError:
            # Occurs if all values in both arrays are identical
            pass

    df_stats = pd.DataFrame(results)
    
    # 4. Correct for Multiple Testing (FDR)
    if not df_stats.empty:
        _, p_adj, _, _ = multipletests(df_stats['P_Value'], method='fdr_bh')
        df_stats['FDR_Adjusted_P'] = p_adj
        
        # Sort by statistical significance
        df_stats = df_stats.sort_values('FDR_Adjusted_P')
        
    return df_stats

def print_top_differential_rules(df_stats, group_a, top_n=10):
    if df_stats.empty:
        return
        
    # Filter for significant rules (FDR < 0.05) driven by Group A (Positive Mean Difference)
    sig_group_a = df_stats[(df_stats['FDR_Adjusted_P'] < 0.05) & (df_stats['Mean_Difference'] > 0)]
    
    print(f"\n--- Top {top_n} Rules Significantly Enriched in {group_a} ---")
    if sig_group_a.empty:
        print("No statistically significant rules found.")
    else:
        display_cols = ['Rule', f'Mean_{group_a}', 'Mean_Difference', 'FDR_Adjusted_P']
        display(sig_group_a.head(top_n)[display_cols])


In [ ]:
# Example Analysis: What rules define Severe Pathological cases compared to Control?
df_stats_severe_vs_control = run_differential_analysis(
    feature_matrix=feature_matrix,
    metadata_df=df_fovs,
    target_col='Pathological score',
    group_a='Severe',
    group_b='Control'
)

# Show rules defining Severe
print_top_differential_rules(df_stats_severe_vs_control, group_a='Severe')

# Show rules defining Control
print_top_differential_rules(df_stats_severe_vs_control, group_a='Control')


Running Differential Analysis: Severe vs Control (based on Pathological score)

--- Top 10 Rules Significantly Enriched in Severe ---


,Rule,Mean_Severe,Mean_Difference,FDR_Adjusted_P
3,CD4T -> Goblet,0.624426,0.916263,2.575495e-11
104,Plasma -> Goblet,0.375438,0.588985,8.627072e-06
108,Plasma -> Neuron,0.145654,0.855530,1.234278e-04
51,Goblet -> CD4T,0.366924,0.504913,5.492073e-04
107,Plasma -> Muscle,0.266534,0.836500,1.770578e-03
64,Goblet -> Plasma,0.145624,0.048167,7.226483e-03
58,Goblet -> Mast,0.234290,0.238307,9.160846e-03
35,Epithelial -> SMV,0.301466,0.536610,1.243242e-02
84,Muscle -> Goblet,0.281303,0.567698,1.243242e-02
67,Goblet -> Treg,-0.044219,0.506370,1.554889e-02



--- Top 10 Rules Significantly Enriched in Control ---


,Rule,Mean_Control,Mean_Difference,FDR_Adjusted_P
3,CD4T -> Goblet,-0.291837,0.916263,2.575495e-11
104,Plasma -> Goblet,-0.213547,0.588985,8.627072e-06
108,Plasma -> Neuron,-0.709876,0.855530,1.234278e-04
51,Goblet -> CD4T,-0.137989,0.504913,5.492073e-04
107,Plasma -> Muscle,-0.569966,0.836500,1.770578e-03
64,Goblet -> Plasma,0.097457,0.048167,7.226483e-03
58,Goblet -> Mast,-0.004017,0.238307,9.160846e-03
35,Epithelial -> SMV,-0.235144,0.536610,1.243242e-02
84,Muscle -> Goblet,-0.286395,0.567698,1.243242e-02
67,Goblet -> Treg,-0.550588,0.506370,1.554889e-02


: 